# 05 — Model Training: PCOS Symptoms Dataset

**Goal:** Train multiple classifiers to predict **PCOS Risk Level** (Low / Medium / High) from symptom data.

| Step | What | Why |
|------|------|-----|
| 1 | Load processed data | Already encoded + SMOTE-balanced from FE notebook |
| 2 | Define risk level target | Map binary PCOS + feature intensity → 3-class risk |
| 3 | Train 3 models | LR, RF, XGBoost — breadth covers linear + non-linear |
| 4 | Evaluate with 4 metrics | Accuracy, F1 (macro), ROC-AUC, Confusion Matrix |
| 5 | Cross-validate best models | Guard against lucky train/test split |
| 6 | Save all trained models | Evaluation notebook loads these for comparison |

In [2]:
import pandas as pd
import numpy as np
import joblib
import os
import warnings
warnings.filterwarnings("ignore")

# ── Sklearn ────────────────────────────────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report
)
from sklearn.model_selection import cross_val_score, StratifiedKFold
from xgboost import XGBClassifier # pip install xgboost

os.makedirs("../models/symptoms", exist_ok=True)
print("Libraries loaded ✓")

Libraries loaded ✓


## 1. Load Processed Data

In [3]:
X_train = pd.read_csv("../data/processed/symptoms_X_train.csv")
y_train_raw = pd.read_csv("../data/processed/symptoms_y_train.csv").squeeze()
X_test  = pd.read_csv("../data/processed/symptoms_X_test.csv")
y_test_raw  = pd.read_csv("../data/processed/symptoms_y_test.csv").squeeze()

print(f"X_train shape : {X_train.shape}")
print(f"X_test shape  : {X_test.shape}")
print(f"Features      : {list(X_train.columns)}")

X_train shape : (830, 30)
X_test shape  : (137, 30)
Features      : ['Family_background', 'LH_hormone', 'FSH_hormone', 'Diabetes_measurment', 'TSH_hormone', 'Prolactin_hormone', 'Hemoglobin_level', 'Cyst_ovary', 'Diagnosis_age', 'Overweight', 'Period_type', 'Hormonal_imbalance', 'Gain_weight', 'Excess_facial_hair', 'Excess_body_hair', 'Dark_area', 'Pimple_face', 'Hormonal_acne_face', 'Blood_pressure', 'Fast_food', 'Losing_hair', 'Mood_swing_period', 'Craving_PT', 'Depress', 'Mental_stress', 'Insomnia', 'Slow_activity', 'Weight(kg)', 'Height(m)', 'BMI(kg/m*m)']


## 2. Build Risk Level Target (3-class)

The raw PCOS label is binary (0/1). We derive a richer **risk level** using:
- **Low** → PCOS = 0 (no PCOS)
- **Medium** → PCOS = 1 but mild symptom burden (fewer high-signal features active)
- **High** → PCOS = 1 with heavy symptom burden (many high-signal features active)

High-signal features (from Cramér's V EDA): `Period_type`, `Excess_facial_hair`, `Excess_body_hair`, `Dark_area`, `Hormonal_imbalance`, `Cyst_ovary`, `Overweight`, `Gain_weight`, `Insomnia`, `Losing_hair`.
       

In [5]:
# High-signal symptom columns (confirmed by Cramér's V in EDA)
HIGH_SIGNAL_COLS = [
    col for col in [
            "Period_type",
            "Excess_facial_hair",
            "Excess_body_hair",
            "Dark_area",
            "Hormonal_imbalance",
            "Cyst_ovary",
            "Overweight",
            "Gain_weight",
            "Insomnia",
            "Losing_hair",
    ] if col in X_train.columns
]


# For each patient, the code counts how many of these high-signal symptoms are present 

def assign_risk(X, y_binary):
    """
    0 (Low)    → PCOS negative
    1 (Medium) → PCOS positive, symptom burden score < median
    2 (High)   → PCOS positive, symptom burden score >= median
    """
    y_binary = np.array(y_binary)
    # X[HIGH_SIGNAL_COLS]: Symptom burden: count of high-signal features that are activated (>0)
    # selects only the high-signal symptom columns (like Excess facial hair, Cyst ovary, Insomnia).
    #.gt(0): checks if each symptom is present (>0 → True).
    #.sum(axis=1): counts how many symptoms are present per patient → burden score.
    # Example: If a patient has 4 of these symptoms, their burden = 4.
    burden = X[HIGH_SIGNAL_COLS].gt(0).sum(axis=1).values

    # Compute median burden only on PCOS-positive cases
    # If there are PCOS-positive cases, compute the median burden only on them (since severity matters only for positives).
    pcos_mask = y_binary == 1
    if pcos_mask.sum() > 0:
        median_burden = np.median(burden[pcos_mask])
    else:
        median_burden = np.median(burden)

    risk = np.where(
        y_binary == 0, 0,                              # No PCOS → Low
        np.where(burden >= median_burden, 2, 1)        # PCOS : (Burden ≥ median → High Risk (2)) (Burden < median → Medium Risk (1).)
    )
    return risk

y_train = assign_risk(X_train, y_train_raw)
y_test  = assign_risk(X_test,  y_test_raw)

label_names = ["Low Risk", "Medium Risk", "High Risk"]

print("Risk Level Distribution:")
print(f"  TRAIN → Low: {(y_train==0).sum()}  Medium: {(y_train==1).sum()}  High: {(y_train==2).sum()}")
print(f"  TEST  → Low: {(y_test==0).sum()}   Medium: {(y_test==1).sum()}   High: {(y_test==2).sum()}")

Risk Level Distribution:
  TRAIN → Low: 415  Medium: 181  High: 234
  TEST  → Low: 33   Medium: 36   High: 68


## 3. Define Models

| Model | Why include it |
|-------|---------------|
| Logistic Regression | Fast linear baseline; interpretable coefficients |
| Random Forest | Handles non-linearity; built-in feature importance |
| XGBoost | Best gradient boosting; usually top performer |

In [8]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000, class_weight="balanced", random_state=42, C=1.0
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200, class_weight="balanced",
        max_depth=None, min_samples_leaf=2, random_state=42, n_jobs=-1
    ),
    "XGBoost": XGBClassifier(
        n_estimators=200, learning_rate=0.05, max_depth=5,
        use_label_encoder=False, eval_metric="mlogloss",
        random_state=42, n_jobs=-1
    ),
}
print(f"{len(models)} models defined ✓")

3 models defined ✓


## 4. Train & Evaluate All Models

In [9]:
results = {}

# Binarize y_test for multi-class ROC-AUC
y_test_bin = label_binarize(y_test, classes=[0, 1, 2])

for name, model in models.items():
    print(f"\n{'─'*55}")
    print(f"  Training: {name}")
    print(f"{'─'*55}")

    model.fit(X_train, y_train)
    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)

    acc     = accuracy_score(y_test, y_pred)
    f1_mac  = f1_score(y_test, y_pred, average="macro", zero_division=0)
    f1_wt   = f1_score(y_test, y_pred, average="weighted", zero_division=0)
    roc_auc = roc_auc_score(y_test_bin, y_proba, multi_class="ovr", average="macro")
    cm      = confusion_matrix(y_test, y_pred)

    results[name] = {
        "model":    model,
        "accuracy": round(acc,    4),
        "f1_macro": round(f1_mac, 4),
        "f1_weighted": round(f1_wt, 4),
        "roc_auc":  round(roc_auc,4),
        "cm":       cm,
    }

    print(f"  Accuracy   : {acc:.4f}")
    print(f"  F1 Macro   : {f1_mac:.4f}")
    print(f"  F1 Weighted: {f1_wt:.4f}")
    print(f"  ROC-AUC    : {roc_auc:.4f}")
    print(f"\n  Confusion Matrix (Low / Medium / High):")
    print(pd.DataFrame(cm, index=label_names, columns=label_names).to_string())
    print(f"\n  Classification Report:")
    print(classification_report(y_test, y_pred, target_names=label_names, zero_division=0))


───────────────────────────────────────────────────────
  Training: Logistic Regression
───────────────────────────────────────────────────────
  Accuracy   : 0.8248
  F1 Macro   : 0.7939
  F1 Weighted: 0.8288
  ROC-AUC    : 0.9514

  Confusion Matrix (Low / Medium / High):
             Low Risk  Medium Risk  High Risk
Low Risk           24            9          0
Medium Risk         7           27          2
High Risk           0            6         62

  Classification Report:
              precision    recall  f1-score   support

    Low Risk       0.77      0.73      0.75        33
 Medium Risk       0.64      0.75      0.69        36
   High Risk       0.97      0.91      0.94        68

    accuracy                           0.82       137
   macro avg       0.80      0.80      0.79       137
weighted avg       0.84      0.82      0.83       137


───────────────────────────────────────────────────────
  Training: Random Forest
──────────────────────────────────────────────────

## 5. Summary Table

In [10]:
summary = pd.DataFrame([
    {
        "Model":        name,
        "Accuracy":     r["accuracy"],
        "F1 Macro":     r["f1_macro"],
        "F1 Weighted":  r["f1_weighted"],
        "ROC-AUC":      r["roc_auc"],
    }
    for name, r in results.items()
]).sort_values("ROC-AUC", ascending=False).reset_index(drop=True)

print("\n=== SYMPTOMS — All Models Summary (sorted by ROC-AUC) ===")
print(summary.to_string(index=False))
print(f"\n→ Best model: {summary.iloc[0]['Model']}")


=== SYMPTOMS — All Models Summary (sorted by ROC-AUC) ===
              Model  Accuracy  F1 Macro  F1 Weighted  ROC-AUC
      Random Forest    0.8540    0.8325       0.8581   0.9750
            XGBoost    0.8394    0.8137       0.8435   0.9654
Logistic Regression    0.8248    0.7939       0.8288   0.9514

→ Best model: Random Forest


## 6. Cross-Validation on Top-2 Models

In [11]:
# Combine train+test for CV (use full data, stratified)
X_all = pd.concat([X_train, X_test], ignore_index=True)
y_all = np.concatenate([y_train, y_test])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
top2 = summary["Model"].values[:2]

print("5-Fold Cross-Validation (F1 Macro) — Top 2 Models:")
for name in top2:
    model = results[name]["model"]
    scores = cross_val_score(model, X_all, y_all, cv=cv, scoring="f1_macro", n_jobs=-1)
    print(f"  {name:<22} mean={scores.mean():.4f}  std={scores.std():.4f}  folds={np.round(scores, 4)}")

5-Fold Cross-Validation (F1 Macro) — Top 2 Models:
  Random Forest          mean=0.8829  std=0.0195  folds=[0.8576 0.9008 0.91   0.876  0.8699]
  XGBoost                mean=0.8749  std=0.0171  folds=[0.8546 0.8643 0.9022 0.8861 0.8674]


## 7. Save All Trained Models

In [12]:
for name, r in results.items():
    safe_name = name.lower().replace(" ", "_")
    path = f"../models/symptoms/{safe_name}.pkl"
    joblib.dump(r["model"], path)
    print(f"  Saved: {path}")

# Save risk label mapping for inference
risk_meta = {
    "label_names":      label_names,
    "high_signal_cols": HIGH_SIGNAL_COLS,
    "n_classes":        3,
}
joblib.dump(risk_meta, "../models/symptoms/risk_meta.pkl")

# Save summary for evaluation notebook
summary.to_csv("../models/symptoms/training_summary.csv", index=False)

print("\nAll models saved ✓")
print(f"Best model: {summary.iloc[0]['Model']} (ROC-AUC={summary.iloc[0]['ROC-AUC']})")

  Saved: ../models/symptoms/logistic_regression.pkl
  Saved: ../models/symptoms/random_forest.pkl
  Saved: ../models/symptoms/xgboost.pkl

All models saved ✓
Best model: Random Forest (ROC-AUC=0.975)


## Summary

```
Task    : Multi-class Risk Level (Low / Medium / High)
Dataset : Symptoms — encoded + SMOTE balanced
Models  : Logistic Regression, Random Forest, XGBoost
Metrics : Accuracy, F1 Macro, F1 Weighted, ROC-AUC (OVR macro)

Saved artifacts:
  ../models/symptoms/logistic_regression.pkl
  ../models/symptoms/random_forest.pkl
  ../models/symptoms/xgboost.pkl
  ../models/symptoms/svm.pkl
  ../models/symptoms/mlp.pkl
  ../models/symptoms/risk_meta.pkl
  ../models/symptoms/training_summary.csv

```